# ICP — align two point clouds

**Track B · 3D & Neural Rendering** (also lessons D1–D3) · the core of registration & SLAM front-ends.

**Iterative Closest Point**: match each source point to its nearest target point, solve the best rigid transform (Kabsch/SVD), apply, repeat. We recover a known transform from noisy data.

> CPU is fine.

In [ ]:
import os, math, torch, matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
ITERS = int(os.environ.get("STEPS", 30))
# a 3D shape (two linked rings)
t = torch.linspace(0, 2 * math.pi, 400, device=device)
src = torch.cat([torch.stack([torch.cos(t), torch.sin(t), 0 * t], 1),
                 torch.stack([1 + torch.cos(t), 0 * t, torch.sin(t)], 1)], 0)

## 1 · Make a target = rigidly-moved, noisy copy

In [ ]:
def Rz(a): return torch.tensor([[math.cos(a), -math.sin(a), 0], [math.sin(a), math.cos(a), 0], [0, 0, 1]], device=device)
R_true = Rz(0.7) @ torch.tensor([[1., 0, 0], [0, math.cos(0.4), -math.sin(0.4)], [0, math.sin(0.4), math.cos(0.4)]], device=device)
t_true = torch.tensor([0.6, -0.4, 0.3], device=device)
tgt = (R_true @ src.T).T + t_true + 0.01 * torch.randn_like(src)

## 2 · ICP — nearest neighbours + Kabsch, iterated

In [ ]:
def kabsch(P, Q):
    pc, qc = P.mean(0), Q.mean(0)
    U, S, Vt = torch.linalg.svd((P - pc).T @ (Q - qc))
    d = torch.sign(torch.det(Vt.T @ U.T))
    R = Vt.T @ torch.diag(torch.tensor([1., 1., d], device=device)) @ U.T
    return R, qc - R @ pc
P = src.clone(); hist = []
for it in range(ITERS + 1):
    idx = torch.cdist(P, tgt).argmin(1); Q = tgt[idx]
    R, tt = kabsch(P, Q); P = (R @ P.T).T + tt
    rmse = (P - Q).norm(dim=1).mean().item(); hist.append((it, rmse))
    if it % max(1, ITERS // 10) == 0: print(f"iter {it:3d}  RMSE {rmse:.4f}")

## 3 · Compare — before vs. after alignment

In [ ]:
fig = plt.figure(figsize=(10, 4))
for i, (Pp, ttl) in enumerate([(src, "before"), (P, "after ICP")]):
    ax = fig.add_subplot(1, 2, i + 1, projection="3d")
    ax.scatter(*tgt.cpu().T, s=3, c="C0", label="target"); ax.scatter(*Pp.detach().cpu().T, s=3, c="C3", label="source")
    ax.set_title(ttl); ax.legend(); ax.set_axis_off()
plt.tight_layout(); plt.show()

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/B_icp_registration/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/B_icp_registration"; os.makedirs(run, exist_ok=True)
torch.save({"R": R.detach().cpu(), "t": tt.detach().cpu()}, f"{run}/transform.pt")
json.dump({"rmse": hist}, open(f"{run}/metrics.json", "w"), indent=2)
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

## (Optional) Publish to the Hugging Face Hub
Upload this run as a **model repo** — the checkpoint, `metrics.json` (full loss/eval history) and the results figure, embedded in an auto-generated model card. Do it for each lab, then group them into a **Collection** on your HF profile (Profile → New collection), or with the commented `add_collection_item` call below. It needs a **write token**, so it only runs once you sign in.

In [ ]:
# (optional) publish this run as a Hugging Face model repo — checkpoint + metrics + plot
!pip -q install huggingface_hub
from huggingface_hub import HfApi, notebook_login
import os
notebook_login()   # paste a WRITE token from https://huggingface.co/settings/tokens
api = HfApi(); user = api.whoami()["name"]
lab = os.path.basename(run); repo_id = f"{user}/" + lab.lower().replace("_", "-")
fig = "\n![results](figure.png)\n" if os.path.exists(f"{run}/figure.png") else ""
open(f"{run}/README.md", "w").write("---\ntags: [ropedia-academy, education]\n---\n# " + lab + "\n\nTrained in **Ropedia Academy** (educational lab). Checkpoint, full loss/eval history (metrics.json) and the results figure are included." + fig)
api.create_repo(repo_id, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=run, repo_id=repo_id, commit_message="Upload from Ropedia Academy")
print("uploaded ->", "https://huggingface.co/" + repo_id)
# group everything into one Collection (run once, after a few uploads):
# from huggingface_hub import create_collection, add_collection_item
# col = create_collection("Ropedia Academy - trained models", namespace=user, exists_ok=True)
# add_collection_item(col.slug, item_id=repo_id, item_type="model")

## How this links to tracks A–D
Registration is the front-end of **D · Scene / world** SLAM & fusion.

### Where to go next
- Add point-to-plane ICP, outlier rejection, and a coarse global init (FPFH / RANSAC).
- ICP is the registration step in TSDF fusion and dense SLAM (next labs in track D).